# 10.5 怎麼學到可用的視覺特徵？


一個入口能把紅方塊變成16個向量，還不表示其中保留了可以讀出的顏色與形狀。我們可以先給它一件小任務：兩張圖片一張紅方塊、一張藍圓形，讓它預測各自編號。預測錯的代價會沿計算返回，告訴像素變換哪些數字應該調整。這是視覺特徵開始獲得任務意義的來源。

前置是[10.3的像素變換](https://birdhackor.github.io/tiny-perceptron-vlm/10.3.html)、[1.8分類猜錯代價](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)與[W.6梯度](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)。encoder是編碼器，負責把原始圖變成特徵；classifier是分類頭，把特徵轉成每種候選的分數。交叉熵CE比較分數與正確編號，梯度是這些數字對代價的敏感程度。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from tiny_perceptron.multimodal import VisionEncoder, scene

torch.manual_seed(0)
encoder = VisionEncoder(width=8)
classifier = nn.Linear(8, 2)
images = torch.stack([scene("red", "square"), scene("blue", "circle")])
features = encoder(images)
logits = classifier(features.mean(1))
loss = F.cross_entropy(logits, torch.tensor([0, 1]))
loss.backward()
print("特徵", tuple(features.shape), "分數", tuple(logits.shape))
print("入口收到梯度", encoder.projection.weight.grad.norm().item() > 0)

輸入兩張合成圖，由`stack`堆成一批。`VisionEncoder(width=8)`依序切patch、做線性變換、加位置並處理特徵，輸出`(2,16,8)`。`mean(1)`把每張圖16個位置平均成8個值，再讓分類頭輸出`(2,2)`，即兩張圖各有兩個類別分數。正確編號[0,1]來自我們生成圖片時的設定，不是模型猜出的標籤。

`backward()`把代價的敏感程度傳回所有參與計算的可訓練參數；最後一行檢查入口矩陣梯度長度大於0，正常輸出True。這裡沒有執行參數更新，不能稱為已訓練成功。即使反復訓練這兩張圖，它還可能只是記住兩張樣本，或用顏色代替形狀，因為紅總是方塊、藍總是圓。

真正驗證要有更多顏色與形狀組合，並保留未訓練的組合或生成位置作為測試。之後可拿掉分類頭、保留編碼器特徵，連接語言模型；任務改變也可能需要繼續調整。

這兩張圖的編號0、1只是兩個候選的名字，尚未讓任務單獨辨別「圓形」。想教形狀，資料要有紅圓、藍圓都標圓、紅方、藍方都標方；想教顏色，則相同顏色不同形狀共用顏色標籤。把任務寫清後，才能判斷平均16個位置的特徵是否夠用。例如只問顏色常可以平均亮度，問左右位置卻可能因平均而丟失排列。這一節的分類頭是建立可核對監督的工具，不保證同一個平均表示足以完成之後每一種視覺問題。

練習只把正確編號[0,1]換成[1,0]，先說明新目標在教相反的映射，再執行確認仍能收到梯度。梯度能通過不代表標籤正確，這是準備資料時不能省略人工核對的原因。
